## Risk Management (Take Profit/Stop Loss)

Risk management is handled by [`TripleBarrierConfig`](hummingbot/strategy_v2/executors/position_executor/data_types.py) passed to each [`PositionExecutorConfig`](hummingbot/strategy_v2/executors/position_executor/data_types.py):

From `get_executor_config()`:
- **Take Profit**: Defined in `triple_barrier_config.take_profit` (percentage or fixed amount)
- **Stop Loss**: Defined in `triple_barrier_config.stop_loss`
- **Trailing Stop**: Optional trailing stop mechanism
- **Time Limit**: Exit after a set duration

Each position executor monitors these conditions and closes positions when triggered.


Here’s how PMM Dynamic uses the triple-barrier risk model (per filled order/position):

- When a quote fills, a PositionExecutor starts with triple_barrier_config from your PMMDynamicControllerConfig. The controller’s MACD/NATR logic no longer drives exits; the executor does.

- The three barriers (any one can exit the position):
  - Take Profit (price barrier up for longs, down for shorts)
    - Trigger when unrealized return reaches or exceeds take_profit.
    - Return is computed vs entry using the current market price from the data provider.
  - Stop Loss (price barrier down for longs, up for shorts)
    - Trigger when unrealized drawdown reaches or exceeds stop_loss (in absolute terms).
  - Time Limit (time barrier)
    - Trigger when holding time exceeds time_limit; the executor exits regardless of PnL.

- Optional trailing stop (if configured):
  - After price moves favorably beyond an activation threshold, the stop level “ratchets” with price (never loosens).
  - If price pulls back by trailing_delta from the best favorable move, the stop is hit and the executor exits.

- Side-aware logic:
  - Long: pnl = (price/entry - 1). TP if pnl >= take_profit; SL if pnl <= -stop_loss.
  - Short: pnl = (entry/price - 1). TP if pnl >= take_profit; SL if pnl <= -stop_loss.

- Lifecycle per level:
  - Entry: quote fills → start executor with timestamp, entry_price, size, side, leverage, triple_barrier_config.
  - Monitoring: on each tick, compute PnL, update trailing stop (if any), check TP/SL/time barriers.
  - Exit: when any barrier triggers, cancel related entry orders (if any remain) and place an exit order to close. Executor finishes.

- Configuration knobs (set in triple_barrier_config):
  - take_profit: fraction (e.g., 0.01 = 1%)
  - stop_loss: fraction (e.g., 0.005 = 0.5%)
  - time_limit: duration (e.g., seconds)
  - optional trailing fields (activation threshold and trailing delta)
  - leverage and order type come from the executor/controller config.

- Interaction with PMM Dynamic quoting:
  - Unfilled quotes are continuously re-priced by MACD/NATR logic; triple barrier does not apply until a fill creates a position.
  - Each filled level is managed independently with its own barriers.

Tip: Inspect TripleBarrierConfig in data_types.py to see the exact field names and defaults used in your build.

In [ ]:
import polars as pl

data = pl.read_parquet("../data/1m/1m.parquet").lazy()

In [ ]:
symbol = "ETHUSDT"
start_date = pl.datetime(2023, 1, 1)
end_date = pl.datetime(2024, 1, 1)
pcs = 2

In [ ]:
data = data.filter(
    (pl.col("symbol") == symbol)
    & (pl.col("open_time") >= start_date)
    & (pl.col("open_time") < end_date)
)

信号计算全部基于最近完结的K线，计算结果quote再后移进行回测

In [ ]:
# prepare the metrics
macd_signal_period = 9
macd_fast = 21
macd_slow = 42
macd_window = 500


macd = pl.col("close").ewm_mean(span=macd_fast) - pl.col("close").ewm_mean(
    span=macd_slow
)
macdh = macd - macd.ewm_mean(span=macd_signal_period)
macd_signal = -((macd - macd.rolling_mean(macd_window,min_samples=1)) / macd.rolling_std(macd_window,min_samples=1))
macdh_signal = macdh.sign()
tr = pl.max_horizontal(
    pl.col("high") - pl.col("low"),
    (pl.col("high") - pl.col("close").shift(1)).abs(),
    (pl.col("low") - pl.col("close").shift(1)).abs(),
)
atr = tr.rolling_mean(window_size=14)
natr = atr / pl.col("close")
max_price_shift = natr / 2
mul = (0.5 * macd_signal + 0.5 * macdh_signal) * max_price_shift
ref = pl.col("close") * (1 + mul)

# generate the quotes
## assume there's 3 layers of quotes
spreads = [1,2,4]
spreads = [s * natr for s in spreads]
quote_a1 = (ref * (1 - spreads[0])).round(pcs).shift(1)
quote_a2 = (ref * (1 - spreads[1])).round(pcs).shift(1)
quote_a3 = (ref * (1 - spreads[2])).round(pcs).shift(1)
quote_b1 = (ref * (1 + spreads[0])).round(pcs).shift(1)
quote_b2 = (ref * (1 + spreads[1])).round(pcs).shift(1)
quote_b3 = (ref * (1 + spreads[2])).round(pcs).shift(1)

In [ ]:
inventory_a = (
    pl.when(quote_a1.is_between(pl.col("low"),pl.col("open"),closed="both")).then(1).otherwise(0)
    + pl.when(quote_a2.is_between(pl.col("low"),pl.col("open"),closed="both")).then(1).otherwise(0)
    + pl.when(quote_a3.is_between(pl.col("low"),pl.col("open"),closed="both")).then(1).otherwise(0)
)
inventory_b = (
    pl.when(quote_b1.is_between(pl.col("open"),pl.col("high"),closed="both")).then(1).otherwise(0)
    + pl.when(quote_b2.is_between(pl.col("open"),pl.col("high"),closed="both")).then(1).otherwise(0)
    + pl.when(quote_b3.is_between(pl.col("open"),pl.col("high"),closed="both")).then(1).otherwise(0)
)
bought_cash = (
    pl.when(quote_a1.is_between(pl.col("low"),pl.col("open"),closed="both")).then(quote_a1).otherwise(0)
    + pl.when(quote_a2.is_between(pl.col("low"),pl.col("open"),closed="both")).then(quote_a2).otherwise(0)
    + pl.when(quote_a3.is_between(pl.col("low"),pl.col("open"),closed="both")).then(quote_a3).otherwise(0)
)
sold_cash = (
    pl.when(quote_b1.is_between(pl.col("open"),pl.col("high"),closed="both")).then(quote_b1).otherwise(0)
    + pl.when(quote_b2.is_between(pl.col("open"),pl.col("high"),closed="both")).then(quote_b2).otherwise(0)
    + pl.when(quote_b3.is_between(pl.col("open"),pl.col("high"),closed="both")).then(quote_b3).otherwise(0)
)
balance = (sold_cash - bought_cash).cum_sum()
total_inventory = (inventory_a - inventory_b).cum_sum()
pnl = balance + total_inventory * pl.col("close")

In [ ]:
data.with_columns(total_inventory=total_inventory, pnl=pnl).select(
    "total_inventory", "open_time"
).group_by_dynamic("open_time", every="1d").agg(
    pl.col("total_inventory").last()
).collect().plot.line(x="open_time", y="total_inventory")

In [ ]:
data.with_columns(pnl=pnl).select(
    "pnl", "open_time"
).group_by_dynamic("open_time", every="1d").agg(
    pl.col("pnl").last()
).collect().plot.line(x="open_time", y="pnl")

In [ ]:
inventory_a = pl.when(quote_a1.is_between(pl.col("low"),pl.col("open"),closed="both")).then(1).otherwise(0)
inventory_b = pl.when(quote_b1.is_between(pl.col("open"),pl.col("high"),closed="both")).then(1).otherwise(0)

bought_cash = pl.when(quote_a1.is_between(pl.col("low"),pl.col("open"),closed="both")).then(quote_a1).otherwise(0)
sold_cash = pl.when(quote_b1.is_between(pl.col("open"),pl.col("high"),closed="both")).then(quote_b1).otherwise(0)
balance = (sold_cash - bought_cash).cum_sum()
total_inventory = (inventory_a - inventory_b).cum_sum()
pnl = balance + total_inventory * pl.col("close")

In [ ]:
data.with_columns(total_inventory=total_inventory, pnl=pnl).select(
    "total_inventory", "open_time"
).group_by_dynamic("open_time", every="1d").agg(
    pl.col("total_inventory").last()
).collect().plot.line(x="open_time", y="total_inventory")

In [ ]:
data.with_columns(pnl=pnl).select(
    "pnl", "open_time"
).group_by_dynamic("open_time", every="1d").agg(
    pl.col("pnl").last()
).collect().plot.line(x="open_time", y="pnl")